This Notebook is for Linear Regression experiments

In [2]:
def predict(x, b0, b1):
    return b0+b1*x

In [3]:
def cost(self):
    return sum((self.predict(x)-y)**2 for x, y in zip(self.x, self.y))/len(self.x)

In [4]:
def grad_func(self, i):
    if i==0:
        return sum(2*(self.predict(x)-y) for x, y in zip(self.x, self.y))/len(self.x)
    else:
        return sum(2*(self.predict(x)-y)*x for x, y in zip(self.x, self.y))/len(self.x)

In [5]:
def update_coeff(self, i):
    grad = grad_func(self, i)
    if i==0:
        self.b0 -= self.alpha*grad
    else :
        self.b1 -= self.alpha*grad

Now lets make a whole class

In [8]:
class LinearRegressor:
    def __init__(self, x, y, alpha = 0.01, b0=0, b1=0):
        if len(x) != len(y):
            return TypeError("x and y should have same number of rows.")
        self.x = x
        self.y = y
        self.alpha = alpha
        self.b0 = b0
        self.b1 =b1
        self.i = 0

    def predict(self, x):
        return self.b0+self.b1*x

    def cost(self):
        return sum((self.predict(x)-y)**2 for x, y in zip(self.x, self.y))/len(self.x)

    def grad_func(self, i):
        if i==0:
            return sum(2*(self.predict(x)-y) for x, y in zip(self.x, self.y))/len(self.x)
        else:
            return sum(2*(self.predict(x)-y)*x for x, y in zip(self.x, self.y))/len(self.x)

    def update_coeff(self, i):
        grad = grad_func(self, i)
        if i==0:
            self.b0 -= self.alpha*grad
        else :
            self.b1 -= self.alpha*grad

    def stop_iteration(self, max_epochs = 50):
        self.i += 1
        if self.i>=max_epochs:
            return

    def fit(self):
        self.i = 0
        while not self.stop_iteration():
            self.update_coeff(0)
            self.update_coeff(1)

Now I will be doing implmentation of linear regression using PyTorch

In [1]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset 
import matplotlib.pyplot as plt

In [2]:
device = torch.device("cuda")

In [3]:
bs = 2
lr = 1e-2
epochs = 100

In [4]:
x = torch.tensor([
    [1.0],
    [2.0],
    [3.0],
    [4.0],
    [5.0],
    [6.0],
    [7.0],
    [8.0],
    [9.0],
    [10.0]
])

y = torch.tensor([
    [3.0],
    [5.0],
    [7.0],
    [9.0],
    [11.0],
    [13.0],
    [15.0],
    [17.0],
    [19.0],
    [21.0]
])

In [5]:
x = x.to(device)
y = y.to(device)

In [6]:
x.shape

torch.Size([10, 1])

In [7]:
x

tensor([[ 1.],
        [ 2.],
        [ 3.],
        [ 4.],
        [ 5.],
        [ 6.],
        [ 7.],
        [ 8.],
        [ 9.],
        [10.]], device='cuda:0')

In [8]:
n = len(x)
train_size = int(0.7*n)
valid_size = int(0.85*n)
x_train = x[:train_size]
y_train = y[:train_size]

x_val = x[train_size:valid_size]
y_val = y[train_size:valid_size]

x_test = x[valid_size:]
y_test = y[valid_size:]

In [9]:
mean = x_train.mean(dim=0)
std = x_train.std(dim=0)

x_train = (x_train-mean)/std
x_val = (x_val-mean)/std
x_test = (x_test-mean)/std

Broadcasting

In [10]:
train_dataset = TensorDataset(x_train, y_train)
train_dataloader = DataLoader(train_dataset, 
                              batch_size = bs, 
                              shuffle = True)

In [11]:
torch.cuda.is_available()

True

In [12]:
class LinearRegression(nn.Module):

    def __init__(self, input_features):
        super().__init__()

        self.linear = nn.Linear(input_features, 1)

    def forward(self, x):
        return self.linear(x)


model = LinearRegression(
    input_features=x_train.shape[1]
).to(device)

In [13]:
criterion = nn.MSELoss()
optimizer = torch.optim.SGD(
    model.parameters(),
    lr = lr,
)

In [14]:
next(model.parameters()).device

device(type='cuda', index=0)

In [15]:
model = model.to(device)
model.train()
train_loss = 0
for a in range(epochs):
    train_loss = 0
    for d, t in train_dataloader:
        d.to(device)
        t.to(device)
        preds = model(d)
        loss = criterion(preds, t)
        train_loss += loss
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    print(f"Epoch {a+1}/{epochs} | ",train_loss)

Epoch 1/100 |  tensor(486.3690, device='cuda:0', grad_fn=<AddBackward0>)
Epoch 2/100 |  tensor(402.6823, device='cuda:0', grad_fn=<AddBackward0>)
Epoch 3/100 |  tensor(290.0246, device='cuda:0', grad_fn=<AddBackward0>)
Epoch 4/100 |  tensor(230.3503, device='cuda:0', grad_fn=<AddBackward0>)
Epoch 5/100 |  tensor(220.7386, device='cuda:0', grad_fn=<AddBackward0>)
Epoch 6/100 |  tensor(165.5339, device='cuda:0', grad_fn=<AddBackward0>)
Epoch 7/100 |  tensor(150.4230, device='cuda:0', grad_fn=<AddBackward0>)
Epoch 8/100 |  tensor(129.2607, device='cuda:0', grad_fn=<AddBackward0>)
Epoch 9/100 |  tensor(115.6992, device='cuda:0', grad_fn=<AddBackward0>)
Epoch 10/100 |  tensor(91.8051, device='cuda:0', grad_fn=<AddBackward0>)
Epoch 11/100 |  tensor(81.7220, device='cuda:0', grad_fn=<AddBackward0>)
Epoch 12/100 |  tensor(67.3140, device='cuda:0', grad_fn=<AddBackward0>)
Epoch 13/100 |  tensor(71.4993, device='cuda:0', grad_fn=<AddBackward0>)
Epoch 14/100 |  tensor(64.7865, device='cuda:0', gr

In [16]:
x_train.shape

torch.Size([7, 1])

In [17]:
model_2 = LinearRegression(
    input_features = x_train.shape[1]
).to(device)

In [18]:
bs_2 = 32

In [19]:
optimizer_2 = torch.optim.SGD(
    model_2.parameters(),
    lr = lr
)
train_dataloader_2 = DataLoader(
    train_dataset,
    batch_size = bs_2,
    shuffle = True
)

In [20]:
model_2.train()
train_loss_2 = 0
for a in range(epochs):
    train_loss_2 = 0
    for u, v in train_dataloader_2:
        u = u.to(device)
        v = v.to(device)
        optimizer.zero_grad()
        preds = model_2(u)
        loss = criterion(preds, v)
        loss.backward()
        optimizer_2.step()
        train_loss_2 += loss
    print(f"Epochs {a+1}/{epochs} | ", train_loss_2)

Epochs 1/100 |  tensor(86.8028, device='cuda:0', grad_fn=<AddBackward0>)
Epochs 2/100 |  tensor(83.4363, device='cuda:0', grad_fn=<AddBackward0>)
Epochs 3/100 |  tensor(76.9665, device='cuda:0', grad_fn=<AddBackward0>)
Epochs 4/100 |  tensor(67.8990, device='cuda:0', grad_fn=<AddBackward0>)
Epochs 5/100 |  tensor(56.9422, device='cuda:0', grad_fn=<AddBackward0>)
Epochs 6/100 |  tensor(44.9520, device='cuda:0', grad_fn=<AddBackward0>)
Epochs 7/100 |  tensor(32.8646, device='cuda:0', grad_fn=<AddBackward0>)
Epochs 8/100 |  tensor(21.6235, device='cuda:0', grad_fn=<AddBackward0>)
Epochs 9/100 |  tensor(12.1052, device='cuda:0', grad_fn=<AddBackward0>)
Epochs 10/100 |  tensor(5.0512, device='cuda:0', grad_fn=<AddBackward0>)
Epochs 11/100 |  tensor(1.0097, device='cuda:0', grad_fn=<AddBackward0>)
Epochs 12/100 |  tensor(0.2931, device='cuda:0', grad_fn=<AddBackward0>)
Epochs 13/100 |  tensor(2.9535, device='cuda:0', grad_fn=<AddBackward0>)
Epochs 14/100 |  tensor(8.7793, device='cuda:0', gr

Model_1 : final train loss = 1e-4
Model_2 : final training loss = 10.3108

In [21]:
model.eval()
with torch.no_grad():
    x_val = x_val.to(device)
    y_val = y_val.to(device)
    preds = model(x_val)
    loss = criterion(preds, y_val)
    print(f"valid loss : {loss:.4f}")

valid loss : 0.0001


valid loss of model 1 = 1e-4

In [23]:
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

In [24]:
total_params

2

In [27]:
model.eval()
with torch.no_grad():
    x_val = x_val.to(device)
    y_val = y_val.to(device)
    preds = model(x_val)
    loss = criterion(preds, y_val)
    print(loss)

tensor(0.0001, device='cuda:0')


In [28]:
model.eval()
with torch.no_grad():
    x_test = x_test.to(device)
    y_test = y_test.to(device)
    preds = model(x_test)
    loss = criterion(preds, y_test)
    print(loss)

tensor(0.0002, device='cuda:0')
